# TFP-BD 80/20 training: EfficientDet-D3 (2 GPU)

Amar nijer **80/20 split** (80% train 18,942 / 10% val / 10% test = team er same 2,368 test chobi), nijer data preparation (original XML -> label).

Ager model theke jana: tf_efficientdet_d3, 9 class, **640 x 640, 30 epoch**. Ager training er baki setting (optimizer, lr) pawa jayni, tai standard EfficientDet fine-tuning: COCO weight, AdamW lr 2e-4, warm-up + cosine, horizontal flip, total batch 16, mixed precision. **Report e eta lekha thakbe.**

**Input:** Bangladeshi Traffic Flow Dataset (nijer) + Data Preparation (Sharmin, shudhu common test list). **GPU T4 x2, Internet ON.**

**Cell 3 = SMOKE TEST:** choto 48 ta chobi diye 2 step training + checkpoint + resume check kore. CPU teo chole (GPU quota lage na),
tai GPU dewar AGE Accelerator **None** diye Cell 1-3 chalie "SMOKE OK" dekhe nite paro.

Asol run: GPU T4 x2 -> **Save Version -> Save & Run All**. Somoy: ~8-10 ghonta (2 x T4). Laptop off thakleo cholbe.
Shesh e output e `effdet_d3_8020_FINAL.pth` thakbe -> Evaluation notebook e ei notebook er output Input hisebe dibe.

In [ ]:
# CELL 1: setup + input khuje ber kora
!pip install -q effdet
import os, time, json, glob, shutil, random, subprocess, sys
T0 = time.time()
import torch
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPU:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)] or "(nai - CPU)")
WK = "/kaggle/working"

SH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            SH = c
            break
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "Bangladeshi Traffic Flow Dataset")]
assert SH, "tfp_dataset pawa jayni - Add Input e Sharmin er 'Data Preparation' notebook jog koro (shudhu test-er file naam er jonno)"
ORIG = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "Bangladeshi Traffic Flow Dataset" in dirs:
        c = os.path.join(root, "Bangladeshi Traffic Flow Dataset")
        if os.path.isdir(os.path.join(c, "Annotated Images")):
            ORIG = c
            break
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "tfp_dataset")]
assert ORIG, "Original 'Bangladeshi Traffic Flow Dataset' pawa jayni - Add Input e jog koro"
print("Original dataset (nijer):", ORIG)
print("Common test list:", SH + "/test")

# resume: shudhu jodi ager run er output (je khane effdet_8020_ckpt.pth ache) Input e jog kora thake
RESUME = ""
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "tfp_dataset", "Bangladeshi Traffic Flow Dataset", "tfp_8010")]
    if "effdet_8020_ckpt.pth" in files:
        RESUME = os.path.join(root, "effdet_8020_ckpt.pth")
print("Resume checkpoint:", RESUME or "nai (notun training shuru hobe)")

In [ ]:
%%writefile /kaggle/working/train_effdet_8020.py
# EfficientDet-D3 on my 80/20 split.
# Known from the team's 70/20/10 model: tf_efficientdet_d3, 9 classes, 640x640 input, 30 epochs,
# ImageNet normalisation, class ids 1..9 (that is how the old evaluation decodes it).
# The rest of the old training recipe is not available, so a standard EfficientDet fine-tuning recipe is used:
# COCO-pretrained weights, AdamW lr 2e-4, weight decay 1e-4, warm-up then cosine decay, horizontal flip,
# total batch 16 (8 per GPU on 2 GPUs), gradient clipping 10, mixed precision. Checkpoint every epoch,
# time limit with resume.
import os, time, math, random
import numpy as np
import torch
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from PIL import Image
from effdet import create_model

DATA = os.environ["DATA_ROOT"]
OUT = os.environ["OUT_DIR"]
EPOCHS = int(os.environ.get("EPOCHS", "30"))
BATCH = int(os.environ.get("BATCH", "16"))
LR = float(os.environ.get("LR", "2e-4"))
SIZE = int(os.environ.get("IMG", "640"))
DEADLINE = float(os.environ.get("DEADLINE", "1e18"))
RESUME = os.environ.get("RESUME", "")
SMOKE = int(os.environ.get("SMOKE", "0"))
MAX_IT = int(os.environ.get("MAX_IT", "0"))
WORKERS = int(os.environ.get("WORKERS", "2"))
CKPT = os.path.join(OUT, os.environ.get("CKPT_NAME", "effdet_8020_ckpt.pth"))
FINAL = os.path.join(OUT, os.environ.get("FINAL_NAME", "effdet_d3_8020_FINAL.pth"))
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

DDP = int(os.environ.get("WORLD_SIZE", "1")) > 1
CUDA = torch.cuda.is_available()
if DDP:
    dist.init_process_group("nccl" if CUDA else "gloo")
    RANK, WORLD, LOCAL = dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", "0"))
else:
    RANK, WORLD, LOCAL = 0, 1, 0
DEV = torch.device(f"cuda:{LOCAL}" if CUDA else "cpu")
if CUDA:
    torch.cuda.set_device(DEV)
MAIN = RANK == 0


def log(*a):
    if MAIN:
        print(*a, flush=True)


class EffDetDataset(Dataset):
    """YOLO labels -> EfficientDet targets: 640x640 resize (same as the old evaluation), boxes in yxyx pixels, class 1..9."""

    def __init__(self, img_dir, label_dir, size=640, train=True):
        self.img_dir, self.label_dir, self.size, self.train = img_dir, label_dir, size, train
        imgs = sorted(f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))
        self.imgs = [f for f in imgs if os.path.getsize(self._lp(f)) > 1] if os.path.isdir(label_dir) else imgs

    def _lp(self, f):
        return os.path.join(self.label_dir, f.rsplit(".", 1)[0] + ".txt")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        f, S = self.imgs[i], self.size
        im = Image.open(os.path.join(self.img_dir, f)).convert("RGB").resize((S, S), Image.BILINEAR)
        a = (np.asarray(im, dtype=np.float32) / 255.0 - MEAN) / STD
        boxes, cls = [], []
        with open(self._lp(f)) as fh:
            for line in fh:
                q = line.split()
                if len(q) != 5:
                    continue
                c, xc, yc, w, h = int(float(q[0])), *map(float, q[1:5])
                x0, y0 = max(0.0, (xc - w / 2) * S), max(0.0, (yc - h / 2) * S)
                x1, y1 = min(float(S), (xc + w / 2) * S), min(float(S), (yc + h / 2) * S)
                if x1 - x0 < 1 or y1 - y0 < 1:
                    continue
                boxes.append([y0, x0, y1, x1])
                cls.append(c + 1)
        if self.train and random.random() < 0.5:
            a = a[:, ::-1, :]
            boxes = [[y0, S - x1, y1, S - x0] for y0, x0, y1, x1 in boxes]
        img = torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).contiguous()
        return img, torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4), torch.tensor(cls, dtype=torch.float32)


def collate(batch):
    imgs = torch.stack([b[0] for b in batch])
    n = max(1, max(len(b[2]) for b in batch))
    bb = torch.zeros(len(batch), n, 4)
    cc = torch.full((len(batch), n), -1.0)
    for i, (_, b, c) in enumerate(batch):
        bb[i, :len(c)] = b
        cc[i, :len(c)] = c
    return imgs, {"bbox": bb, "cls": cc}


ds = EffDetDataset(f"{DATA}/train/images", f"{DATA}/train/labels", SIZE, train=True)
sampler = DistributedSampler(ds, num_replicas=WORLD, rank=RANK, shuffle=True, seed=0) if DDP else None
PER_GPU = max(1, BATCH // WORLD)
dl = DataLoader(ds, batch_size=PER_GPU, shuffle=sampler is None, sampler=sampler, num_workers=WORKERS,
                pin_memory=CUDA, collate_fn=collate, drop_last=True)
NB = len(dl)

bench = create_model("tf_efficientdet_d3", bench_task="train", num_classes=9, pretrained=not RESUME,
                     image_size=(SIZE, SIZE), bench_labeler=True)
start_epoch, ck = 0, None
if RESUME and os.path.exists(RESUME):
    ck = torch.load(RESUME, map_location="cpu", weights_only=False)
    bench.model.load_state_dict(ck["model_state_dict"])
    start_epoch = int(ck["epoch"])
    log(f"RESUME: {RESUME} -> {start_epoch} epoch already done")
bench.to(DEV)
net = torch.nn.parallel.DistributedDataParallel(bench, device_ids=[LOCAL] if CUDA else None) if DDP else bench

optimizer = torch.optim.AdamW(bench.parameters(), lr=LR, weight_decay=1e-4)
TOTAL, WARM = EPOCHS * NB, min(500, NB)


def lr_lambda(step):
    if step < WARM:
        return 0.1 + 0.9 * step / WARM
    p = (step - WARM) / max(1, TOTAL - WARM)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=CUDA)
if ck is not None:
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    scheduler.load_state_dict(ck["scheduler_state_dict"])
    if CUDA and "scaler_state_dict" in ck:
        scaler.load_state_dict(ck["scaler_state_dict"])
    del ck

log(f"train images {len(ds)} | GPUs {WORLD if CUDA else 0} | batch per GPU {PER_GPU} (total {PER_GPU * WORLD}) | "
    f"iterations per epoch {NB} | image {SIZE} | mixed precision {CUDA} | epochs {start_epoch + 1}..{EPOCHS}")


def state(epoch):
    return {"epoch": epoch, "model_state_dict": bench.model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(), "scaler_state_dict": scaler.state_dict(),
            "config": {"model": "tf_efficientdet_d3", "num_classes": 9, "image_size": SIZE, "class_ids": "1..9"}}


stopped, last = False, start_epoch
for epoch in range(start_epoch, EPOCHS):
    if sampler is not None:
        sampler.set_epoch(epoch)
    net.train()
    t_ep, run_loss, n_it = time.time(), 0.0, 0
    for it, (imgs, tgt) in enumerate(dl):
        imgs = imgs.to(DEV, non_blocking=True)
        tgt = {k: v.to(DEV, non_blocking=True) for k, v in tgt.items()}
        with torch.autocast(device_type="cuda" if CUDA else "cpu", enabled=CUDA):
            out = net(imgs, tgt)
            loss = out["loss"]
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(bench.parameters(), 10.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        run_loss += float(loss.item())
        n_it += 1
        if it == 0 or (it + 1) % 200 == 0:
            el = time.time() - t_ep
            log(f"epoch {epoch + 1}/{EPOCHS}  it {it + 1}/{NB}  loss {loss.item():.4f}  lr {scheduler.get_last_lr()[0]:.2e}  |  "
                f"{el / 60:.1f} min, this epoch ~{el / (it + 1) * NB / 60:.0f} min")
        if MAX_IT and it + 1 >= MAX_IT:
            break
    ep_time = time.time() - t_ep
    last = epoch + 1
    log(f"Epoch {epoch + 1} Completed | Average Loss: {run_loss / max(n_it, 1):.4f} | {ep_time / 60:.1f} min")
    if MAIN:
        torch.save(state(epoch + 1), CKPT)
    stop = torch.tensor([1 if (epoch + 1 < EPOCHS and time.time() + ep_time * 1.1 > DEADLINE) else 0], device=DEV)
    if DDP:
        dist.broadcast(stop, 0)
    if int(stop.item()):
        log(f"TIME LIMIT: {epoch + 1}/{EPOCHS} epoch done, checkpoint saved -> resume needed")
        stopped = True
        break
    if SMOKE and epoch + 1 - start_epoch >= 1:
        break

if MAIN and not stopped and last == EPOCHS:
    torch.save(state(EPOCHS), FINAL)
    log(f"FINAL saved: {FINAL}")
if DDP:
    dist.barrier()
    dist.destroy_process_group()

In [ ]:
# CELL 3: data preparation helper + SMOKE TEST (choto data diye pura pipeline check)

# ---- NIJER data preparation (original XML theke YOLO label) + 80/10/10 split
# test = team er common test set (70/20/10 er same 2,368 chobi, shudhu file-er NAAM Sharmin er folder theke pora hoy)
# train/val = baki 21,310 chobi theke NIJER random split (seed 42): val 2,368 (10%), train 18,942 (80%)
import random, glob
import xml.etree.ElementTree as ET
EXTS = (".jpg", ".jpeg", ".png", ".bmp")
SPLIT_ROOT = "/kaggle/working/tfp_8010"
CLS9 = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
NAME_MAP = {"car": "Car", "bus": "Bus", "truck": "Truck", "bike": "Motorcycle", "cycle": "Bicycle",
            "rickshaw": "Rickshaw", "mini-truck": "Mini-Truck", "cng": "CNG", "people": "Pedestrian"}

def voc_to_yolo_lines(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    W, H = float(size.find("width").text), float(size.find("height").text)
    lines = []
    for obj in root.findall("object"):
        key = str(obj.find("name").text).strip().lower()
        if key not in NAME_MAP:
            continue
        b = obj.find("bndbox")
        x0, y0, x1, y1 = [float(b.find(t).text) for t in ("xmin", "ymin", "xmax", "ymax")]
        lines.append(f"{CLS9.index(NAME_MAP[key])} {((x0 + x1) / 2) / W:.6f} {((y0 + y1) / 2) / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
    return lines

def build_split_8010(SH, out_root=SPLIT_ROOT, make_links=True):
    # 1) original dataset theke image + XML jora
    pairs = {}
    for xf in glob.glob(os.path.join(ORIG, "**", "*.xml"), recursive=True):
        st = os.path.splitext(os.path.basename(xf))[0]
        for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
            ip = os.path.join(os.path.dirname(xf), st + ext)
            if os.path.exists(ip):
                pairs[st] = (ip, xf)
                break
    # 2) common test set (shudhu naam)
    test = sorted(os.path.splitext(f)[0] for f in os.listdir(f"{SH}/test/images") if f.lower().endswith(EXTS))
    assert all(t in pairs for t in test), "test chobi original dataset e pawa jayni"
    pool = sorted(set(pairs) - set(test))
    order = pool[:]
    random.Random(42).shuffle(order)
    n_val = len(test)
    S = {"train": sorted(order[n_val:]), "val": sorted(order[:n_val]), "test": test}
    if make_links:
        for s, stems in S.items():
            os.makedirs(f"{out_root}/{s}/images", exist_ok=True)
            os.makedirs(f"{out_root}/{s}/labels", exist_ok=True)
            for st in stems:
                ip, xf = pairs[st]
                dst = f"{out_root}/{s}/images/{os.path.basename(ip)}"
                if not os.path.lexists(dst):
                    os.symlink(ip, dst)
                with open(f"{out_root}/{s}/labels/{st}.txt", "w") as fh:
                    fh.write("\n".join(voc_to_yolo_lines(xf)) + "\n")
    tot = sum(len(v) for v in S.values())
    print(f"Original dataset: {len(pairs)} image+XML jora")
    print("Nijer 80/10/10 split:", {k: f"{len(v)} ({len(v) / tot * 100:.1f}%)" for k, v in S.items()})
    # 3) sanity: amar conversion ar Sharmin er label ki eki? (200 ta test file)
    if make_links:
        same, chk = 0, test[:200]
        for st in chk:
            mine = open(f"{out_root}/test/labels/{st}.txt").read().split()
            hers = open(f"{SH}/test/labels/{st}.txt").read().split()
            same += mine == hers
        print(f"  Label conversion check: {same}/{len(chk)} file Sharmin er conversion er sathe hubohu mile")
    return S, pairs

SM = f"{WK}/smoke"
shutil.rmtree(SM, ignore_errors=True)
for d in ("data/train/images", "data/train/labels", "out"):
    os.makedirs(f"{SM}/{d}")
n = 0
for root, dirs, files in os.walk(ORIG):
    for f in sorted(files):
        if not f.endswith(".xml") or n >= 48:
            continue
        st = f[:-4]
        img = next((os.path.join(root, st + e) for e in (".jpg", ".jpeg", ".png") if os.path.exists(os.path.join(root, st + e))), None)
        lines = voc_to_yolo_lines(os.path.join(root, f))
        if img and lines:
            os.symlink(img, f"{SM}/data/train/images/{os.path.basename(img)}")
            open(f"{SM}/data/train/labels/{st}.txt", "w").write("\n".join(lines) + "\n")
            n += 1
    if n >= 48:
        break
print("smoke chobi:", n)
NP_SMOKE = 1 if NGPU == 1 else 2   # CPU te-o 2 process (gloo) diye 2-GPU code path test hoy

def torchrun(env_extra, nproc):
    env = dict(os.environ, **env_extra)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}", f"{WK}/train_effdet_8020.py"]
    return subprocess.run(cmd, env=env).returncode

base = {"DATA_ROOT": f"{SM}/data", "OUT_DIR": f"{SM}/out", "SMOKE": "1", "EPOCHS": "2", "MAX_IT": "2",
        "BATCH": "4", "WORKERS": "0"}
t = time.time()
rc1 = torchrun(base, NP_SMOKE)
assert rc1 == 0 and os.path.exists(f"{SM}/out/effdet_8020_ckpt.pth"), "SMOKE step 1 FAILED - upore error dekho, screenshot pathao"
rc2 = torchrun(dict(base, RESUME=f"{SM}/out/effdet_8020_ckpt.pth"), NP_SMOKE)
assert rc2 == 0 and os.path.exists(f"{SM}/out/effdet_d3_8020_FINAL.pth"), "SMOKE step 2 (resume) FAILED - screenshot pathao"
shutil.rmtree(SM, ignore_errors=True)
print(f"SMOKE OK ({time.time() - t:.0f}s): training, checkpoint ar resume sob kaj kore")

In [ ]:
# CELL 4: amar 80/20 split (original XML theke, symlink - disk e copy hoy na) ~15-20 min
S, SRC = build_split_8010(SH)
json.dump(S, open(f"{WK}/split_8010_lists.json", "w"))
print("train", len(S["train"]), "| val", len(S["val"]), "| test", len(S["test"]))

In [ ]:
# CELL 5: asol training (~8-10 ghonta (2 x T4))
assert NGPU >= 1, "GPU nai - Session options e Accelerator: GPU T4 x2 dao. CPU te training hoy na (shudhu smoke test hoy)."
NP = 2 if NGPU >= 2 else 1
DEADLINE = T0 + 11.2 * 3600          # Kaggle 12 ghonta limit er age checkpoint rekhe thambe
env = {"DATA_ROOT": SPLIT_ROOT, "OUT_DIR": WK, "EPOCHS": "30", "BATCH": "16", "WORKERS": "2", "DEADLINE": str(DEADLINE)}
if RESUME:
    shutil.copy(RESUME, f"{WK}/resume_from.pth")
    env["RESUME"] = f"{WK}/resume_from.pth"
print(f"{NP} GPU diye training shuru | somoy-shima: aro {(DEADLINE - time.time()) / 3600:.1f} ghonta")
rc = torchrun(env, NP)
print("return code:", rc)

In [ ]:
# CELL 6: fol
if os.path.exists(f"{WK}/resume_from.pth"):
    os.remove(f"{WK}/resume_from.pth")
if os.path.exists(f"{WK}/effdet_d3_8020_FINAL.pth"):
    print("TRAINING SHESH: effdet_d3_8020_FINAL.pth ready.")
    print("Ekhon Evaluation notebook e ei notebook er output Input hisebe jog kore Save & Run All dao.")
else:
    ep = torch.load(f"{WK}/effdet_8020_ckpt.pth", map_location="cpu", weights_only=False)["epoch"] if os.path.exists(f"{WK}/effdet_8020_ckpt.pth") else 0
    print(f"Training shesh hoyni: {ep}/30 epoch hoyeche. Screenshot pathao - resume er niyom bole dibo.")
print("Output:", sorted(f for f in os.listdir(WK) if f.endswith((".pth", ".json"))))